# 📊 Notebook 01: Data Preprocessing Pipeline
## AI Health Risk Predictor — Cardiovascular Disease

This notebook walks through the full data preprocessing pipeline:
1. **Dataset Generation** — Synthetic EHR with clinically plausible correlations
2. **Exploratory Data Analysis** — Distributions, correlations, missing data
3. **Data Cleaning** — Handling missing values and outliers
4. **Feature Engineering** — Derived clinical features
5. **Train/Val/Test Split** — Stratified splitting
6. **Scaling & Encoding** — Preparing data for ML


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 100
print('Setup complete ✓')

## 1. Generate Synthetic EHR Dataset

In [ ]:
from src.data_generator import generate_synthetic_ehr

df = generate_synthetic_ehr(n_patients=5000, seed=42)

Path('../data').mkdir(parents=True, exist_ok=True)
df.to_csv('../data/synthetic_ehr.csv', index=False)

print(f'Dataset shape: {df.shape}')
print(f'Event prevalence: {df["event_within_5yrs"].mean():.1%}')
df.head()

## 2. Exploratory Data Analysis

In [ ]:
# Missing data summary
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing N': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing N'] > 0]
print('Missing values:')
display(missing_df)

# Statistical summary
numeric_cols = ['age','systolic_bp','diastolic_bp','total_chol','hdl','ldl','hba1c','bmi']
df[numeric_cols].describe().T.round(2)

In [ ]:
# Distribution plots for numeric features
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, col in zip(axes.flatten(), numeric_cols):
    df[col].dropna().hist(bins=40, ax=ax, color='#3498db', edgecolor='white', alpha=0.85)
    ax.set_title(col.replace('_', ' ').title(), fontweight='bold')
    ax.set_ylabel('Count')
fig.suptitle('Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Event prevalence by key groups
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col in zip(axes, ['sex', 'smoking_status', 'ethnicity']):
    prev = df.groupby(col)['event_within_5yrs'].mean().sort_values(ascending=False)
    prev.plot.bar(ax=ax, color='#e74c3c', edgecolor='white', alpha=0.85)
    ax.set_title(f'Event Rate by {col.replace("_", " ").title()}')
    ax.set_ylabel('5-Year Event Rate')
    ax.set_ylim(0, 0.4)
    for p in ax.patches:
        ax.annotate(f'{p.get_height():.1%}', (p.get_x()+p.get_width()/2, p.get_height()+0.005),
                    ha='center', va='bottom', fontsize=9)
    ax.tick_params(axis='x', rotation=25)

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
corr = df[numeric_cols + ['event_within_5yrs', 'diabetes_flag']].corr()
fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr), k=1)
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, ax=ax, linewidths=0.5)
ax.set_title('Feature Correlation Heatmap', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Age by event status
fig, ax = plt.subplots(figsize=(7, 4))
df.groupby('event_within_5yrs')['age'].plot.kde(ax=ax, legend=True)
ax.set_xlabel('Age')
ax.set_title('Age Distribution by CVD Event Status')
ax.legend(['No Event', 'CVD Event'])
plt.tight_layout()
plt.show()

## 3. Run Preprocessing Pipeline

In [ ]:
from src.preprocessor import run_preprocessing_pipeline

splits = run_preprocessing_pipeline(
    input_csv='../data/synthetic_ehr.csv',
    output_dir='../data/splits',
    preprocessor_path='../models/preprocessor.pkl',
    seed=42
)

X_train, y_train = splits['X_train'], splits['y_train']
X_val,   y_val   = splits['X_val'],   splits['y_val']
X_test,  y_test  = splits['X_test'],  splits['y_test']

print(f'Feature matrix shape: {X_train.shape}')
print(f'Feature names: {list(X_train.columns[:10])} ...')
print(f'\nTrain prevalence: {y_train.mean():.1%}')
print(f'Val prevalence:   {y_val.mean():.1%}')
print(f'Test prevalence:  {y_test.mean():.1%}')

In [ ]:
# Show engineered features
eng_features = ['pulse_pressure', 'chol_ratio', 'bmi_category', 'hba1c_elevated',
                'med_count', 'bp_elevated', 'is_male', 'age_decade']
available = [f for f in eng_features if f in X_train.columns]

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, feat in zip(axes.flatten(), available):
    X_train[feat].hist(bins=30, ax=ax, color='#9b59b6', edgecolor='white', alpha=0.85)
    ax.set_title(feat.replace('_', ' ').title(), fontweight='bold')
fig.suptitle('Engineered Features (Training Set)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Class Balance Check

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3))
counts = y_train.value_counts()
ax.bar(['No Event', 'CVD Event'], counts.values, 
       color=['#3498db', '#e74c3c'], edgecolor='white', width=0.5)
for i, (v, c) in enumerate(zip(counts.index, counts.values)):
    ax.text(i, c + 10, str(c), ha='center', fontweight='bold')
ax.set_title('Training Set Class Balance')
ax.set_ylabel('Patient Count')
plt.tight_layout()
plt.show()
print(f'Class ratio (neg:pos) = {counts[0]}:{counts[1]} = {counts[0]/counts[1]:.1f}:1')